# 🎮 DQN 入门: 用神经网络玩 CartPole

上一个 notebook 用 Q 表在 4×4 网格里找奶酪——状态只有 16 个，Q 表装得下。

现在换成 **CartPole**：一根杆子立在滑块上，你要控制滑块左右移动，让杆子不倒。

状态是**连续值**（位置、速度、角度、角速度），不能再用 Q 表了。

**用一个小神经网络代替 Q 表**——这就是 DQN（Deep Q-Network）。

## 1. 认识 CartPole 环境

CartPole 是 RL 里最经典的入门环境——"小车上顶一根杆子，控制滑块让杆不倒"。

- **State**（4 个连续值）：滑块位置、滑块速度、杆角度、杆角速度
- **Action**（2 个离散值）：左推 (0) / 右推 (1)
- **Reward**：每坚持 1 步 +1 分（最多 500 步）

In [ ]:
# 安装 gymnasium（OpenAI Gym 的继任者）
# 如果你已有 gymnasium 环境，可以跳过
import sys, subprocess
try:
    import gymnasium as gym
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "gymnasium"])
    import gymnasium as gym
    print("gymnasium 安装成功 ✅")

In [ ]:
import gymnasium as gym

env = gym.make("CartPole-v1")
state, info = env.reset()

print(f"状态空间: {env.observation_space}")  # 4 维连续值
print(f"动作空间: {env.action_space}")       # 2 个离散动作 (左/右)
print(f"初始状态: {state}")                  # [位置, 速度, 角度, 角速度]

## 2. 先看随机策略的表现

没有任何学习——就是随机推左推右。杆子能撑几秒？

In [ ]:
env = gym.make("CartPole-v1", render_mode="rgb_array")
state, info = env.reset()
total_reward = 0

for step in range(500):
    action = env.action_space.sample()  # 随机选动作
    state, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    if terminated or truncated:
        break

print(f"随机策略撑了 {step+1} 步，得了 {total_reward} 分")
print("（一般随机策略只能撑 10-30 步，杆子就倒了）")
env.close()

## 3. DQN：用神经网络代替 Q 表

### 为什么要用神经网络？

Q 表：`Q[状态编号][动作编号]` — 只能在**离散状态**下用。
神经网络：`nn(状态向量) → [动作0的Q值, 动作1的Q值]` — **连续状态也能用**。

### 核心思想（和 Q-Learning 一模一样）：

```
目标Q值 = 奖励 + γ × max(神经网络预测的下一步Q值)
损失 = (神经网络预测的当前Q值 - 目标Q值)²
然后反向传播 → 更新网络参数 → 预测更准
```

### 两个关键技巧：

1. **Replay Buffer**：存过去的经验（状态,动作,奖励,下一状态），随机抽样学习——打破时间相关性，一个经验能反复学
2. **Target Network**：目标网络是主网络的"冻结副本"，每 N 步同步一次——防止"追自己的尾巴"（训练不稳定）

In [ ]:
import torch
import torch.nn as nn
import random
from collections import deque

class DQN(nn.Module):
    """一个小型神经网络：输入4个状态值 → 输出2个动作的Q值"""
    def __init__(self):
        super().__init__()
        # TODO: 设计网络结构
        # 输入 4 个值（状态），输出 2 个值（每个动作的 Q 值）
        # 用 2 层隐藏层就好，ReLU activation
        # self.net = ???
        pass
    
    def forward(self, x):
        return self.net(x)

# 答案：
class DQN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 128),     # 状态 4 维 → 128 个神经元
            nn.ReLU(),
            nn.Linear(128, 128),   # 128 → 128
            nn.ReLU(),
            nn.Linear(128, 2),     # 128 → 2 个动作的 Q 值
        )
    
    def forward(self, x):
        return self.net(x)

print("网络定义完成！参数量:", sum(p.numel() for p in DQN().parameters()))

## 4. Replay Buffer（经验回放池）

把每次交互的经验存起来，学习时随机抽一批——一个经验能反复学，还能打破时间相关性。

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)  # 固定大小，满了自动丢旧的
    
    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))
    
    def sample(self, batch_size=64):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.FloatTensor(states),
            torch.LongTensor(actions),
            torch.FloatTensor(rewards),
            torch.FloatTensor(next_states),
            torch.FloatTensor(dones)
        )
    
    def __len__(self):
        return len(self.buffer)

buffer = ReplayBuffer()
print("Replay Buffer 创建完成 ✅")

## 5. 训练 DQN

把一切串起来——交互 → 存经验 → 抽样 → 更新网络。

In [ ]:
import torch.optim as optim

# 初始化
env = gym.make("CartPole-v1")
policy_net = DQN()          # 主网络（不断更新）
target_net = DQN()          # 目标网络（每 N 步同步）
target_net.load_state_dict(policy_net.state_dict())

optimizer = optim.Adam(policy_net.parameters(), lr=0.001)
buffer = ReplayBuffer(10000)

# 超参数
GAMMA = 0.99                # 折扣因子
BATCH_SIZE = 64             # 每批学习多少条经验
TARGET_UPDATE = 100         # 每多少步同步目标网络
EPSILON_START = 1.0         # 初始完全随机
EPSILON_END = 0.01          # 最终随机 1%
EPSILON_DECAY = 500         # 500 步内从 100% 衰减到 1%

episode_rewards = []
total_steps = 0

for episode in range(200):  # 训练 200 回合
    state, info = env.reset()
    episode_reward = 0
    
    while True:
        # 探索率衰减
        epsilon = EPSILON_END + (EPSILON_START - EPSILON_END) * \
                  max(0, 1 - total_steps / EPSILON_DECAY)
        
        # ε-greedy 选动作
        if random.random() < epsilon:
            action = random.randint(0, 1)
        else:
            with torch.no_grad():
                q_values = policy_net(torch.FloatTensor(state))
                action = q_values.argmax().item()
        
        # 执行动作
        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        episode_reward += reward
        
        # 存入经验池
        buffer.push(state, action, reward, next_state, done)
        state = next_state
        total_steps += 1
        
        # 经验够了才开始学
        if len(buffer) >= BATCH_SIZE:
            # 抽样
            states, actions, rewards, next_states, dones = buffer.sample(BATCH_SIZE)
            
            # 计算目标 Q 值
            with torch.no_grad():
                next_q = target_net(next_states).max(dim=1).values
                target_q = rewards + GAMMA * next_q * (1 - dones)
            
            # 当前 Q 值
            current_q = policy_net(states).gather(1, actions.unsqueeze(1)).squeeze()
            
            # 损失 & 更新
            loss = nn.MSELoss()(current_q, target_q)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        
        if done:
            break
    
    # 同步目标网络
    if episode % TARGET_UPDATE == 0:
        target_net.load_state_dict(policy_net.state_dict())
    
    episode_rewards.append(episode_reward)
    if episode % 20 == 0:
        avg = sum(episode_rewards[-20:]) / len(episode_rewards[-20:])
        print(f"回合 {episode:3d} | 奖励 {episode_reward:4.0f} | 近20局平均 {avg:6.1f} | buffer {len(buffer):5d} | ε {epsilon:.3f}")

env.close()

In [ ]:
# 画训练曲线
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.plot(episode_rewards, alpha=0.3, color='blue', label='每局得分')

# 平滑曲线（滑动平均）
window = 10
smoothed = [sum(episode_rewards[max(0,i-window):i+1]) / min(i+1, window) 
            for i in range(len(episode_rewards))]
plt.plot(smoothed, color='red', linewidth=2, label=f'{window}局滑动平均')

plt.axhline(y=200, color='green', linestyle='--', label='及格线 (200分)')
plt.xlabel('回合')
plt.ylabel('得分')
plt.title('CartPole DQN 训练曲线')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

if smoothed[-1] > 200:
    print("✅ 训练成功！DQN 学会了平衡杆子！")
else:
    print("⚠️ 还没到 200 分，可能需要多训练一些回合")

## 6. 从 CartPole 到 G1：跨越了什么？

| | CartPole DQN | G1 PPO |
|---|---|---|
| 状态维度 | 4 | ~200（29 个关节 × 多个特征 + 5 步历史） |
| 动作维度 | 2（离散：左/右） | 29（连续：每个关节转多少度） |
| 算法 | DQN | PPO（处理连续动作的改进版） |
| 训练环境 | gymnasium（几行代码） | Isaac Sim（完整物理引擎） |

**但核心思想完全一样**：
- 神经网络看状态 → 预测 Q 值（或直接输出动作）
- 和环境交互 → 收集 (state, action, reward) 数据
- 用数据更新神经网络 → 让好的行为更可能发生